# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² colorectal cancer dataset using the `mlcroissant` library, following the Croissant metadata schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Dataset: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
- DOI: [10.71728/senscience.qs2f-h81p](https://doi.org/10.71728/senscience.qs2f-h81p)
- Source: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print('Dataset Title:', metadata.name)
print('Description:', metadata.description)

## 2. Data Overview
Review available record sets, their fields, and `@id`s as described in the Croissant schema.

In [ ]:
# List all record sets and their fields using their @id

record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets discovered in metadata; attempting to retrieve dynamically.")
    # mlcroissant 0.7+: dataset.record_sets returns all record sets discovered
    record_sets = dataset.record_sets
else:
    # In Croissant, recordSet in metadata may be a single dict or list
    if not isinstance(record_sets, list):
        record_sets = [record_sets]

print(f"Number of record sets: {len(record_sets)}\n")

overview = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
    print(f"Record Set @id: {rs_id}")
    try:
        # Discover fields for each record set
        schema = dataset.record_set_schema(record_set=rs_id)
        fields = []
        for f in schema:
            f_id = f['@id'] if '@id' in f else f.get('name', 'unknown')
            # Some Croissant schemas may include columns instead of fields
            fields.append(f_id)
        print(f"  Fields: {fields}\n")
        overview.append({'record_set_id': rs_id, 'fields': fields})
    except Exception as e:
        print(f"  Could not load fields/schema for record set {rs_id}: {e}\n")
        overview.append({'record_set_id': rs_id, 'fields': None})


## 3. Data Extraction
Load data from each record set into pandas DataFrames, referencing every entity by its `@id`. We'll show the available fields and data sample for each set.

_If the dataset has a single main record set, we'll focus on that for the next steps._

In [ ]:
# Gather all record set @ids from overview
rs_ids = [item['record_set_id'] for item in overview if item['fields']]
dataframes = {}

sampled = False
sample_record_set = None
for record_set_id in rs_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields (@id): {list(df.columns)}")
        print(df.head())
        if not sampled and len(df) > 0:
            sample_record_set = record_set_id
            sampled = True
    except Exception as e:
        print(f"  Could not extract data for record set {record_set_id}: {e}\n")

# Select a sample record set for further analysis
if sample_record_set:
    print(f"\nWe'll use record set '{sample_record_set}' for exploratory analysis.")
else:
    raise ValueError("No populated record sets available for EDA.")

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field and a group/categorical field by their `@id` from the record set we chose above. We'll:
1. Filter rows based on a numeric threshold
2. Normalize the numeric column
3. Group by the categorical field and display means

In [ ]:
# --- EDA using variable @id fields and dynamic column selection ---
df = dataframes[sample_record_set]
# You may need to manually choose a numeric and group field based on available columns:

print("Available columns (@id):", df.columns.tolist())

# Attempt to automatically select numeric and group fields (customize if needed)
numeric_field_id = None
group_field_id = None

# Try common heuristics: choose the first float/integer column for numeric, first categorical/text for grouping
for col in df.columns:
    try:
        # Try converting to numeric, skipping missing/non-numeric
        sample_vals = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(sample_vals) > 0:
            numeric_field_id = col
            break
    except Exception:
        continue

# Choose the first field that is object/string and has few unique values (likely a group/categorical field)
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]):
        nunique = df[col].nunique(dropna=True)
        if 1 < nunique < len(df) // 2:
            group_field_id = col
            break
if numeric_field_id is None:
    raise ValueError("No numeric field found to analyze. Please examine column names and types above.")

print(f"Selected numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Selected group field: {group_field_id}")

# Filter records by a threshold on numeric field
try:
    threshold = df[numeric_field_id].astype(float).mean()  # Set threshold as mean for demonstration
except Exception:
    threshold = 10  # fallback

filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()

print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field among filtered entries
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If a group/categorical field is available, group and summarize
if group_field_id is not None:
    # Only group by the filtered rows
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f'{numeric_field_id}_mean')
    )
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field, and optionally grouping by the chosen categorical field (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a group field is available, boxplot grouped by group_field_id
if group_field_id is not None:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded a Croissant-compatible biomedical dataset with `mlcroissant` using only `@id`-based referencing.
- Explored available record sets and their fields.
- Extracted tabular data, selected relevant fields, filtered and normalized a numeric column, and performed group-wise summaries.
- Visualized data distributions and relationships interactively.

This approach can be extended for other datasets using Croissant schemas, supporting reproducible FAIR data analysis workflows.